In [4]:

base_url="./Dataset/"

In [5]:
import numpy as np
import pandas as pd
import nltk

from sklearn.preprocessing import OneHotEncoder
def wordTokenize(text:str):
    return nltk.word_tokenize(text)



In [13]:
# Loading Dataset and Preprocessing
data = pd.read_csv(f'{base_url}go_emotions_dataset.csv')
data.head()
df = data["text"].to_frame(name= "text")

def normalizeData(df):
    for index,row in df.iterrows():
        text = row['text']
        text = text.lower()
        text.strip()
        text.strip('''
        !"#$%&'()*+,-./:;<=>?@[]\\^_`{|}~
        ''')
        row['text'] = text
    return df

df = normalizeData(df)

vocabulary = {
    "<PAD>": 0,
    "<UNK>": 1}

currIdx=2

max_len = 60 # can increase to 80 for better context understanding

def buildVocabulary(df):
    global vocabulary
    global currIdx
    global max_len
    for idx,row in df.iterrows():
        text = row['text']
        text = wordTokenize(text)
        numSequence = []

        for word in text:
            if word not in vocabulary:
                vocabulary[word] = currIdx
                numSequence.append(currIdx)
                currIdx+=1
            else:
                numSequence.append(vocabulary[word])

        numSequence =  np.array(numSequence)
        if(len(numSequence)<max_len):
            padLen = max_len-len(numSequence)
            numSequence = np.append(numSequence ,  [vocabulary["<PAD>"]]*padLen)
        else:
            numSequence = numSequence[0:max_len]

        row['text'] = numSequence

buildVocabulary(pd.DataFrame(data["text"]))
y = data.drop(columns= ["id" , "text"])
print(y.head())
print("Vocabulary Size : " , currIdx)



TypeError: Invalid value for dtype 'str'. Value should be a string or missing value (or array of those).

In [ ]:
print(len(df))
print(df.head())
print(y_df.head())

In [ ]:
from sklearn.model_selection import train_test_split

# X = df.drop(columns = 'emotion_label'  ) # it will be treated as object because pandas will store this column as object even though this contains a sequence of int 64  (the column itself it object type ) and keras required integer type
# print(X)
# X = np.array(df['text'].tolist() , dtype = np.int32) # because keras required the input to be a 2d or 3d tensor and drop column
X = np.stack(df['text'].values).astype(np.int32)
X_train , X_test , y_train , y_test = train_test_split(X , y_df , test_size =0.2 , random_state = 42)

print("success")


In [ ]:
# Before Training -> Let's Save the X and y (preprocessed ), so as to avoid the  preprocessing everytime
import os
from joblib import dump
os.makedirs(f"{base_url}preProcessedInputs" , exist_ok = True)

dump({
    "X_train":X_train,
    "X_test":X_test,
    "y_train":y_train,
    "y_test":y_test,
    "vocab":vocabulary
},f"{base_url}preProcessedInputs/dataset.joblib")

In [ ]:

from joblib import load
import torch
data = load(f'{base_url}preProcessedInputs/dataset.joblib')
X_train= data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
vocab = data["vocab"]

# We have saved y as one hot encoding dataframe -> convert it numpy array
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()
#  One-hot -> class indices
y_train = np.argmax(y_train , axis = 1)
y_test = np.argmax(y_test, axis = 1)
# Converting to Pytorch tensors

X_train= torch.tensor(X_train , dtype = torch.long)
X_test = torch.tensor(X_test , dtype = torch.long)
y_train = torch.tensor(y_train , dtype = torch.long)
y_test = torch.tensor(y_test , dtype = torch.long)
